# Teach the RoArm-M3 to pick up electrodes

1. **Record**: the arm goes limp, you move it by hand through the pick, and it records joint angles + camera frames.
2. **Train**: LeRobot's ACT policy learns to copy you (behavior cloning).
3. **Run**: the policy drives the arm from what the cameras see.

**Safety:** the arm is strong. Keep your face out of its reach, and clamp the base to the table. Stop any cell with the ■ button; the arm locks where it is.

## Settings

In [ ]:
from roarm import teach
from roarm.arm import RoArm
from roarm.camera import Camera, find_cameras

ARM = None                          # None = find it on USB. Or "COM14", or "192.168.1.203" (WiFi, slower)
CAMERAS = {"wrist": 0, "front": 1}  # name -> camera number (see find_cameras() below)
TASK = "pick up the electrode"
FPS = 10
IMAGE_SIZE = (240, 320)             # (height, width) saved per frame

DATASET = "data/electrode_pick"
REPO_ID = "local/electrode_pick"
RUN = "outputs/act_electrode_pick"

## 1. Connect

In [ ]:
print("Cameras found:", find_cameras())

arm = RoArm(ARM)
print("Arm on", arm.target, "| joints (rad):", arm.state().round(2))

cams = {name: Camera(index) for name, index in CAMERAS.items()}
teach.preview(cams, IMAGE_SIZE)

## 2. Record demonstrations

Each demo: the arm goes home, you hold it, it goes limp, you guide it to grab the electrode and lift it, then press Enter.

- Aim for **~50 demos**. Run this cell again to add more; they're appended.
- **Move the electrode to a new spot every time**, so the policy has to look rather than memorize one spot.
- Move smoothly and do it the same way each time. Hold the arm low (upper arm/elbow) so your hand stays out of the wrist camera.
- Open and close the gripper by hand. Answer **n** to throw away a bad demo.

In [ ]:
dataset = teach.open_dataset(DATASET, REPO_ID, FPS, list(cams), IMAGE_SIZE)
try:
    for _ in range(10):
        teach.record_episode(arm, cams, dataset, TASK, FPS, IMAGE_SIZE)
finally:
    dataset.finalize()
print(dataset.num_episodes, "demos saved in", DATASET)

## 3. Train

This takes a while on the GTX 1060. Checkpoints land in `RUN/checkpoints/`. To train again, change `RUN` to a new folder name.

In [ ]:
teach.train(DATASET, REPO_ID, RUN, steps=20_000)

## 4. Let the policy drive

Put an electrode somewhere it can reach. `max_step` limits how far any joint can move per step (radians); keep it small at first.

In [ ]:
CHECKPOINT = f"{RUN}/checkpoints/last/pretrained_model"
teach.run_policy(arm, cams, CHECKPOINT, TASK, FPS, IMAGE_SIZE, seconds=30, max_step=0.1)

## Done

In [ ]:
arm.close()
for cam in cams.values():
    cam.close()